# 🐝 Lab 1.1 — Gradient Boosting From Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varoonsahgal/advancedml/blob/main/notebooks/lab01_boosting_from_scratch.ipynb)

**Gradient Hive: Advanced ML Week 2026 · Day 1 — Classical, Sharpened**

⏱ **Time:** ~2 hours · runs on CPU (Colab free tier is plenty)

### 🎯 Learning objectives
- Decompose test error into **bias², variance and noise** by simulation, and see why bagging and boosting attack different terms.
- Reproduce **double descent** with random-features regression and explain the interpolation peak.
- Implement **gradient boosting from scratch** for squared loss and log loss (Newton leaf updates), matching scikit-learn to machine precision.
- Use **shrinkage, row subsampling and early stopping** to control a boosted ensemble.
- Benchmark your booster against **HistGradientBoosting, LightGBM and XGBoost** on real tabular data and explain what makes them faster.
- Spot **feature-importance pitfalls** (impurity bias, correlated features) and fix them with permutation importance.

### 🧰 Prerequisites
- Python, NumPy, pandas, and basic scikit-learn (`fit`/`predict`).
- Knowing what a decision tree is. Calculus at the level of 'the derivative of a loss'.

### 🌍 Why this matters in 2026
Deep learning gets the headlines, but most production ML on spreadsheets (fraud, credit, churn, ads, logistics) still runs on gradient-boosted trees. Benchmarks such as Grinsztajn et al. (2022) found tuned GBDTs still beating deep nets on typical medium-sized tabular data, and in 2026 they are the baseline every new tabular foundation model (we meet TabPFN in Lab 1.3) has to beat. Writing a booster yourself is also the cleanest way to learn *functional gradient descent*: the same idea of gradient steps, taken in function space instead of parameter space.

> 🐝 *Buzz the Backprop Bee says:* run every cell top to bottom. Each **🧪 Your Turn** cell is followed by a **✅ Solution** cell, so the notebook always runs end to end. Try the exercise first, then peek.

## 🗺️ Roadmap

| Part | What you'll do | ⏱ min |
|---|---|---|
| ⚙️ Setup | install/import, seeds, palette | 3 |
| 1 · Bias–variance | simulate 200 parallel universes; bagging demo | 15 |
| 2 · Double descent | random-features regression past the interpolation threshold | 12 |
| 3 · GB for regression | write the boosting loop; watch residuals get eaten (animation) | 25 |
| 4 · GB for classification | log loss + Newton leaves; evolving decision boundaries | 18 |
| 5 · Regularization trio | shrinkage, subsampling, early stopping on California housing | 20 |
| 6 · Showdown | scratch vs sklearn vs HistGB vs LightGBM vs XGBoost | 10 |
| 7 · Importance pitfalls | trap features, permutation & grouped importance | 15 |
| 🏁 Wrap-up | takeaways, interview corner, boss level | 5 |

## ⚙️ Setup

In [ ]:
# ⚙️ Setup: run me first!
import os, sys, time, math, random, warnings, importlib.util
FAST = os.environ.get("LAB_FAST", "0") == "1"   # smoke-test mode: shrinks everything

def ensure(pkgs):
    """pip-install packages that are missing (LightGBM/XGBoost are preinstalled on Colab)."""
    missing = [pip for pip, mod in pkgs if importlib.util.find_spec(mod) is None]
    if missing:
        print("📦 Installing:", missing)
        get_ipython().run_line_magic("pip", "install -q " + " ".join(missing))

ensure([("lightgbm", "lightgbm"), ("xgboost", "xgboost")])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (GradientBoostingRegressor, GradientBoostingClassifier,
                              HistGradientBoostingRegressor, RandomForestRegressor)
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, log_loss
from sklearn.datasets import make_friedman1, make_moons
warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
N_JOBS = 2          # parallel workers for sklearn (Colab free tier has 2 vCPUs)
random.seed(SEED); np.random.seed(SEED)

HIVE = ["#F5B700", "#1F1B16", "#FF6B5B", "#1FA39A", "#3D4DB7", "#9B5DE5"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=HIVE), "figure.facecolor": "#FFFBEA", "axes.facecolor": "#FFFDF5", "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
print(f"FAST={FAST} | numpy {np.__version__} | python {sys.version.split()[0]}")

---
## Part 1 — Bias, variance, and why ensembles work

**Intuition.** Imagine a hundred parallel universes. In each one you collect a fresh training set from the same process and fit the same model. At a fixed input $x$ you get a hundred slightly different predictions.
* Their **average** may sit away from the truth. That gap is **bias** (the model is too rigid).
* Their **spread** is **variance** (the model is too jumpy and chases noise).

**The math.** With $y = f(x) + \varepsilon$, $\mathbb{E}[\varepsilon]=0$, $\operatorname{Var}(\varepsilon)=\sigma^2$, and $\hat f_D$ trained on a random dataset $D$:

$$
\mathbb{E}_{D,\varepsilon}\big[(y - \hat f_D(x))^2\big] \;=\; \underbrace{\big(\mathbb{E}_D[\hat f_D(x)] - f(x)\big)^2}_{\text{bias}^2} \;+\; \underbrace{\mathbb{E}_D\big[(\hat f_D(x) - \mathbb{E}_D[\hat f_D(x)])^2\big]}_{\text{variance}} \;+\; \underbrace{\sigma^2}_{\text{noise}}
$$

We can't visit parallel universes, but we can **simulate** them because we pick $f$ ourselves. We'll fit regression trees of increasing depth on $B$ independent datasets.

In [ ]:
rng = np.random.default_rng(SEED)

def f_true(x):
    return np.sin(2 * np.pi * x) + 0.5 * x

SIGMA, N_TRAIN = 0.3, 60
x_grid = np.linspace(0, 1, 200)
f_grid = f_true(x_grid)

def sample_dataset(rng, n=N_TRAIN):
    x = rng.uniform(0, 1, n)
    return x, f_true(x) + rng.normal(0, SIGMA, n)

B = 40 if FAST else 200                    # number of "parallel universes"
DEPTHS = [1, 2, 3, 4, 5, 6, 8, 10, 14]
preds = {}                                 # depth -> (B, len(x_grid)) predictions
for depth in DEPTHS:
    P = np.empty((B, len(x_grid)))
    for b in range(B):
        x, y = sample_dataset(rng)
        P[b] = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(x[:, None], y).predict(x_grid[:, None])
    preds[depth] = P
print("Simulated", B * len(DEPTHS), "trees 🌳")

### 🧪 Your Turn 1 — Implement the decomposition
Fill in `bias_variance(P, f_grid)`. `P[b, g]` is model $b$'s prediction at grid point $g$.

*Hints:* `P.mean(axis=0)` is $\mathbb{E}_D[\hat f_D(x)]$ at every grid point; `P.var(axis=0)` is the variance across universes. Average both over the grid.

In [ ]:
# 🧪 Your Turn 1
def bias_variance(P, f_grid):
    """Return (bias^2, variance), each averaged over the grid points."""
    mean_pred = None   # TODO: average prediction across the B models, shape (G,)
    bias2 = None       # TODO: mean over grid of (mean_pred - f_grid)**2
    variance = None    # TODO: mean over grid of the across-model variance
    return bias2, variance

In [ ]:
# ✅ Solution (peek only after trying!)
def bias_variance(P, f_grid):
    """Return (bias^2, variance), each averaged over the grid points."""
    mean_pred = P.mean(axis=0)
    bias2 = np.mean((mean_pred - f_grid) ** 2)
    variance = np.mean(P.var(axis=0))
    return bias2, variance

In [ ]:
# ✔️ Checkpoint: bias² + variance + σ² should equal the empirical test MSE on fresh noisy labels
bv = {d: bias_variance(preds[d], f_grid) for d in DEPTHS}
for d in DEPTHS:
    noisy_y = f_grid + rng.normal(0, SIGMA, preds[d].shape)
    mse_emp = np.mean((preds[d] - noisy_y) ** 2)
    total = bv[d][0] + bv[d][1] + SIGMA ** 2
    assert abs(total - mse_emp) / mse_emp < 0.1, (d, total, mse_emp)
assert bv[1][0] > bv[14][0], "deep trees should have lower bias than stumps"
assert bv[14][1] > bv[1][1], "deep trees should have higher variance than stumps"
print("✔️ Decomposition checks out! bias²+var+σ² ≈ test MSE for every depth.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, d in zip(axes[:2], [1, 14]):
    for b in range(min(25, B)):
        ax.plot(x_grid, preds[d][b], color=HIVE[0], alpha=0.25, lw=1)
    ax.plot(x_grid, preds[d].mean(0), color=HIVE[4], lw=2.5, label="average model")
    ax.plot(x_grid, f_grid, "--", color=HIVE[1], lw=2, label="truth f(x)")
    ax.set_title(f"depth={d}: " + ("high bias, low variance" if d == 1 else "low bias, high variance"))
    ax.legend(loc="lower left"); ax.set_xlabel("x")
b2 = np.array([bv[d][0] for d in DEPTHS]); va = np.array([bv[d][1] for d in DEPTHS])
ax = axes[2]
ax.plot(DEPTHS, b2, "o-", label="bias²", color=HIVE[2])
ax.plot(DEPTHS, va, "o-", label="variance", color=HIVE[3])
ax.plot(DEPTHS, b2 + va + SIGMA**2, "o-", label="expected test MSE", color=HIVE[1])
ax.axhline(SIGMA**2, ls=":", color="gray", label="noise floor σ²")
ax.set_xlabel("tree depth"); ax.set_title("The U-shaped tradeoff"); ax.legend()
plt.tight_layout(); plt.show()
print("Best depth:", DEPTHS[int(np.argmin(b2 + va))])

### Bagging: average away the variance
A practical stand-in for parallel universes is the **bootstrap**: resample your one dataset with replacement. Averaging $K$ trees fit to bootstrap samples (**bagging**, Breiman 1996) leaves bias roughly unchanged. If the trees were independent with variance $v$, the average would have variance $v/K$. In practice they're correlated with pairwise correlation $\rho$, so the variance tends to $\rho v + \frac{1-\rho}{K} v$. Random forests push $\rho$ down by also subsampling features.

> 🔮 **Predict-then-run:** we bag 25 depth-14 trees. Guess: by roughly what factor does variance drop? Does bias go up, down, or stay put?

In [ ]:
def bagged_predict(x, y, n_bags, depth, rng):
    out = np.zeros_like(x_grid)
    for _ in range(n_bags):
        idx = rng.integers(0, len(x), len(x))   # bootstrap resample
        out += DecisionTreeRegressor(max_depth=depth).fit(x[idx, None], y[idx]).predict(x_grid[:, None])
    return out / n_bags

B_BAG, N_BAGS = (15, 10) if FAST else (100, 25)
P_bag = np.array([bagged_predict(*sample_dataset(rng), N_BAGS, 14, rng) for _ in range(B_BAG)])
single = bias_variance(preds[14][:B_BAG], f_grid); bagged = bias_variance(P_bag, f_grid)

fig, ax = plt.subplots(figsize=(6, 3.8))
w = 0.35; xs = np.arange(2)
ax.bar(xs - w/2, single, w, label="single depth-14 tree", color=HIVE[2])
ax.bar(xs + w/2, bagged, w, label=f"bag of {N_BAGS} trees", color=HIVE[3])
ax.set_xticks(xs, ["bias²", "variance"]); ax.legend(); ax.set_title("Bagging attacks variance")
plt.show()
print(f"variance: {single[1]:.4f} -> {bagged[1]:.4f}  (x{single[1]/bagged[1]:.1f} smaller); bias²: {single[0]:.4f} -> {bagged[0]:.4f}")

**Takeaway 🐝.** Bagging takes *low-bias, high-variance* learners (deep trees) and averages them down. **Boosting goes the other way.** It takes *high-bias, low-variance* learners (shallow trees) and adds them up **sequentially**, each one correcting what the ensemble still gets wrong. That drives bias down while shrinkage keeps variance in check. We'll build one from scratch in Part 3.

> **🍯 Key takeaway:** Expected test error = bias² + variance + noise. Bagging averages *variance* away; boosting chips *bias* away.

---
## Part 2 — Double descent: when the U-curve isn't the whole story

The classical picture says that past the sweet spot, more capacity means more overfitting. Modern over-parameterized models (huge nets, big tree ensembles) seem to break this. Belkin et al. (2019) showed that if you keep adding capacity **past the point where the model can interpolate the training data**, test error can **fall again**. That's *double descent*.

**Setup: random-features regression.** Draw $p$ random ReLU features $\phi_j(x) = \max(0, w_j^\top x + b_j)$ with $w_j, b_j$ frozen at random values, then fit only the linear head. With $n$ training points:
* $p < n$: ordinary least squares (under-parameterized).
* $p = n$: there's exactly one interpolating solution, and it's wildly unstable.
* $p > n$: infinitely many interpolators. We take the **minimum-norm** one, $\hat\theta = \Phi^{+} y$ (the pseudo-inverse). Gradient descent from zero converges to this solution too, so the regularization is *implicit*.

$$\hat\theta = \Phi^{+} y = \lim_{\lambda \to 0^+} (\Phi^\top\Phi + \lambda I)^{-1}\Phi^\top y$$

> 🔮 **Predict-then-run:** with $n=40$ training points, sketch the test error as $p$ goes from 2 to 2000. Where's the worst $p$? Does the far right end up better or worse than the best small model?

In [ ]:
def double_descent(n=40, d=5, noise=0.3, n_seeds=20, ridge=0.0):
    ps = np.unique(np.round(np.logspace(0.3, 3.3, 30)).astype(int))
    errs, norms = np.zeros((n_seeds, len(ps))), np.zeros((n_seeds, len(ps)))
    for s in range(n_seeds):
        r = np.random.default_rng(s)
        beta = r.normal(size=d) / np.sqrt(d)
        f = lambda X: np.tanh(2 * X @ beta) + 0.5 * np.sin(3 * X[:, 0])
        Xtr = r.uniform(-1, 1, (n, d)); ytr = f(Xtr) + r.normal(0, noise, n)
        Xte = r.uniform(-1, 1, (1000, d)); yte = f(Xte)
        for j, p in enumerate(ps):
            W, b = r.normal(size=(d, p)), r.normal(size=p)
            phi = lambda X: np.maximum(X @ W + b, 0) / np.sqrt(p)
            Ptr = phi(Xtr)
            if ridge == 0:
                theta = np.linalg.pinv(Ptr, rcond=1e-12) @ ytr          # min-norm least squares
            else:                                                        # ridge in dual form (works for any p)
                theta = Ptr.T @ np.linalg.solve(Ptr @ Ptr.T + ridge * np.eye(n), ytr)
            errs[s, j] = np.mean((phi(Xte) @ theta - yte) ** 2)
            norms[s, j] = np.linalg.norm(theta)
    return ps, np.median(errs, 0), np.median(norms, 0)

N_DD = 40
ps, err_mn, norm_mn = double_descent(n=N_DD, n_seeds=5 if FAST else 20)
_,  err_rg, norm_rg = double_descent(n=N_DD, n_seeds=5 if FAST else 20, ridge=1e-2)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
ax.loglog(ps, err_mn, "o-", color=HIVE[2], label="min-norm (ridgeless)")
ax.loglog(ps, err_rg, "s-", color=HIVE[3], label="ridge λ=0.01")
ax.axvline(N_DD, ls="--", color=HIVE[1]); ax.text(N_DD * 1.08, ax.get_ylim()[1] * 0.5, "p = n\n(interpolation\nthreshold)")
ax.set_xlabel("# random features p"); ax.set_ylabel("test MSE"); ax.set_title("Double descent"); ax.legend()
ax = axes[1]
ax.loglog(ps, norm_mn, "o-", color=HIVE[2], label="min-norm (ridgeless)")
ax.loglog(ps, norm_rg, "s-", color=HIVE[3], label="ridge λ=0.01"); ax.axvline(N_DD, ls="--", color=HIVE[1])
ax.set_xlabel("# random features p"); ax.set_ylabel("‖θ̂‖₂ (median)"); ax.legend()
ax.set_title("Why the peak? The fitted weights explode at p≈n")
plt.tight_layout(); plt.show()
i_peak = np.argmax(err_mn)
print(f"peak at p={ps[i_peak]} (n={N_DD}); best small-p MSE={err_mn[ps < N_DD].min():.3f}, MSE at p={ps[-1]}: {err_mn[-1]:.3f}")

**What just happened?**
1. **Left of the peak** is the classic U: more features help until we start fitting noise.
2. **At $p \approx n$** there's essentially one way to fit all $n$ noisy points exactly. The square feature matrix is ill-conditioned (tiny smallest singular value $\sigma_{\min}$), and $\Phi^+$ amplifies the noise by roughly $1/\sigma_{\min}$. The fitted weights $\|\hat\theta\|$ explode (right plot), and so does the variance.
3. **Right of the peak** there are many interpolators. The min-norm one spreads weight across many features, which acts like an implicit ridge penalty. Its norm, and with it the variance, *shrinks* as $p$ grows.
4. **Explicit ridge** flattens the peak entirely. The peak comes from *unregularized* interpolation, not from capacity as such.

🐝 *Connection to boosting:* boosted ensembles routinely reach ~zero training error and keep improving on test data as you add trees. Classic margin theory (Schapire et al., 1998) and the interpolation view both explain why "more model" isn't automatically "more overfit". Shrinkage and early stopping are still your best friends, as we'll see.

> **🍯 Key takeaway:** The scary peak at $p \approx n$ is an ill-conditioning artifact of unregularized interpolation. Regularization (implicit or explicit) removes it, and heavily over-parameterized models can generalize well.

---
## Part 3 — Gradient boosting for regression, from scratch

**Intuition.** Fit a simple model. Look at what it got wrong (the residuals). Fit another small tree *to the residuals*. Add a fraction of it. Repeat. Each tree is a small step that fixes the current mistakes.

**The math: gradient descent in function space** (Friedman, 2001). We want $F$ minimizing $\sum_i L(y_i, F(x_i))$.
1. Initialize with the best constant: $F_0 = \arg\min_c \sum_i L(y_i, c)$.
2. For $m = 1..M$:
   * compute **pseudo-residuals** $r_{im} = -\left.\frac{\partial L(y_i, F)}{\partial F}\right|_{F = F_{m-1}(x_i)}$,
   * fit a regression tree $h_m$ to $\{(x_i, r_{im})\}$,
   * update $F_m(x) = F_{m-1}(x) + \nu\, h_m(x)$ with **learning rate / shrinkage** $\nu \in (0, 1]$.

For squared loss $L = \tfrac12 (y - F)^2$ we get $-\partial L/\partial F = y - F$, the plain old **residual**, and $F_0 = \bar y$. A squared-error tree's leaf value (the mean of residuals in the leaf) is already the optimal step, so no extra line search is needed.

### 🧪 Your Turn 2 — Write the boosting loop
Complete `gb_fit_regression`. Use `DecisionTreeRegressor(max_depth=max_depth, random_state=0)` as the weak learner.

*Hints:* keep a running vector `F` of current training predictions. Each round: `r = y - F`, fit the tree on `(X, r)`, then `F += lr * tree.predict(X)`.

In [ ]:
# 🧪 Your Turn 2
def gb_fit_regression(X, y, n_estimators=100, lr=0.1, max_depth=3):
    """Return (init, trees) for squared-loss gradient boosting."""
    init = None                          # TODO: best constant for squared loss
    F = np.full(len(y), init)
    trees = []
    for m in range(n_estimators):
        r = None                         # TODO: pseudo-residuals (negative gradient)
        tree = None                      # TODO: fit a DecisionTreeRegressor on (X, r)
        F = None                         # TODO: shrunken update of F
        trees.append(tree)
    return init, trees

def gb_predict_regression(init, trees, lr, X):
    return init + lr * sum(t.predict(X) for t in trees)

In [ ]:
# ✅ Solution (peek only after trying!)
def gb_fit_regression(X, y, n_estimators=100, lr=0.1, max_depth=3):
    """Return (init, trees) for squared-loss gradient boosting."""
    init = float(np.mean(y))
    F = np.full(len(y), init)
    trees = []
    for m in range(n_estimators):
        r = y - F
        tree = DecisionTreeRegressor(max_depth=max_depth, random_state=0).fit(X, r)
        F = F + lr * tree.predict(X)
        trees.append(tree)
    return init, trees

def gb_predict_regression(init, trees, lr, X):
    return init + lr * sum(t.predict(X) for t in trees)

In [ ]:
# ✔️ Checkpoint: match scikit-learn's GradientBoostingRegressor
Xf, yf = make_friedman1(n_samples=500, noise=1.0, random_state=0)
init, trees = gb_fit_regression(Xf, yf, n_estimators=50, lr=0.1, max_depth=3)
ours = gb_predict_regression(init, trees, 0.1, Xf)
sk = GradientBoostingRegressor(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=0).fit(Xf, yf)
diff = np.abs(ours - sk.predict(Xf)).max()
print(f"max |ours - sklearn| = {diff:.2e}")
assert diff < 1e-6, "Your booster should match sklearn's to numerical precision"
print("✔️ Bit-for-bit (almost) identical to sklearn. You just wrote GradientBoostingRegressor! 🍯")

### Watching residuals get eaten, stage by stage
Below, the **top row** shows the ensemble $F_m$ after $m$ rounds. The **bottom row** shows the residuals $r_m = y - F_{m-1}$ that tree $m$ sees, with the tree's (piecewise-constant) fit on top. Residuals start out as the whole sine wave and shrink to structureless noise.

In [ ]:
rng1 = np.random.default_rng(1)
x1 = np.sort(rng1.uniform(0, 1, 150)); y1 = f_true(x1) + rng1.normal(0, SIGMA, 150)
X1, LR1 = x1[:, None], 0.3
init1, trees1 = gb_fit_regression(X1, y1, n_estimators=100, lr=LR1, max_depth=2)
# staged predictions on train points and on the grid
stage_tr = init1 + LR1 * np.cumsum([np.zeros_like(x1)] + [t.predict(X1) for t in trees1], axis=0)
stage_gr = init1 + LR1 * np.cumsum([np.zeros_like(x_grid)] + [t.predict(x_grid[:, None]) for t in trees1], axis=0)

show = [1, 2, 5, 15, 40, 100]
fig, axes = plt.subplots(2, len(show), figsize=(18, 6), sharex=True)
for k, m in enumerate(show):
    ax = axes[0, k]
    ax.scatter(x1, y1, s=8, color="gray", alpha=0.6)
    ax.plot(x_grid, stage_gr[m], color=HIVE[0], lw=2.5); ax.plot(x_grid, f_grid, "--", color=HIVE[1], lw=1)
    ax.set_title(f"F_{m}  (train MSE {np.mean((y1 - stage_tr[m])**2):.3f})", fontsize=10)
    ax = axes[1, k]
    res = y1 - stage_tr[m - 1]
    ax.scatter(x1, res, s=8, color=HIVE[2], alpha=0.7)
    ax.plot(x_grid, trees1[m - 1].predict(x_grid[:, None]), color=HIVE[4], lw=2)
    ax.axhline(0, color="gray", lw=0.5); ax.set_ylim(-2, 2)
    ax.set_title(f"residuals seen by tree {m}", fontsize=10)
axes[0, 0].set_ylabel("ensemble"); axes[1, 0].set_ylabel("residual & tree fit")
plt.tight_layout(); plt.show()

In [ ]:
# 🎬 Optional animation (skipped in FAST mode). It renders as an interactive player in Jupyter/Colab.
if not FAST:
    from matplotlib import animation
    from IPython.display import HTML
    fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 3.6))
    a0.scatter(x1, y1, s=8, color="gray", alpha=0.6); a0.plot(x_grid, f_grid, "--", color=HIVE[1], lw=1)
    line, = a0.plot([], [], color=HIVE[0], lw=2.5); a0.set_ylim(-1.6, 2.0)
    sc = a1.scatter(x1, y1 - stage_tr[0], s=8, color=HIVE[2]); tl, = a1.plot([], [], color=HIVE[4], lw=2)
    a1.set_ylim(-2, 2); a1.axhline(0, color="gray", lw=0.5)
    def draw(m):
        line.set_data(x_grid, stage_gr[m + 1]); a0.set_title(f"ensemble after {m + 1} trees")
        sc.set_offsets(np.c_[x1, y1 - stage_tr[m]]); tl.set_data(x_grid, trees1[m].predict(x_grid[:, None]))
        a1.set_title(f"residuals + tree {m + 1}")
        return line, sc, tl
    anim = animation.FuncAnimation(fig, draw, frames=40, interval=150, blit=False)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))

> **🍯 Key takeaway:** Gradient boosting is gradient descent in *function space*: each tree is a small step along the negative gradient (for squared loss, the residuals).

---
## Part 4 — Boosting for binary classification (log loss + Newton steps)

Now $y \in \{0, 1\}$ and $F(x)$ is a **log-odds** score with $p = \sigma(F) = 1/(1+e^{-F})$. The log loss is

$$L(y, F) = -\big[y \log p + (1-y)\log(1-p)\big], \qquad \frac{\partial L}{\partial F} = p - y, \qquad \frac{\partial^2 L}{\partial F^2} = p(1-p).$$

* **Init:** $F_0 = \log\frac{\bar y}{1-\bar y}$ (log-odds of the base rate).
* **Pseudo-residuals:** $r_i = y_i - p_i$. Fit a regression tree to them.
* **Leaf values:** the tree's default leaf value (mean residual) is *not* optimal for log loss. Friedman's recipe (and sklearn's) replaces each leaf $R_j$'s value with **one Newton step**:

$$\gamma_j = \frac{\sum_{i \in R_j} r_i}{\sum_{i \in R_j} p_i (1 - p_i)} \;=\; -\frac{\sum g_i}{\sum h_i}.$$

This "gradient over Hessian" form is also the heart of XGBoost, which adds an L2 penalty: $\gamma_j = -\frac{G_j}{H_j + \lambda}$. The Hessian also enters XGBoost's split-gain formula.

### 🧪 Your Turn 3 — Newton leaf update
Complete `newton_leaf_update(tree, X, r, p)`. It should overwrite each leaf's value in place.

*Hints:* `leaves = tree.apply(X)` gives the leaf id for every row. Loop over `np.unique(leaves)` and set `tree.tree_.value[leaf, 0, 0] = num / den`. Guard against `den ≈ 0`.

In [ ]:
# 🧪 Your Turn 3
def newton_leaf_update(tree, X, r, p):
    leaves = tree.apply(X)
    for leaf in np.unique(leaves):
        mask = leaves == leaf
        num = None   # TODO: sum of residuals in the leaf
        den = None   # TODO: sum of p*(1-p) in the leaf
        # TODO: tree.tree_.value[leaf, 0, 0] = ...
    return tree

In [ ]:
# ✅ Solution (peek only after trying!)
def newton_leaf_update(tree, X, r, p):
    leaves = tree.apply(X)
    for leaf in np.unique(leaves):
        mask = leaves == leaf
        num = r[mask].sum()
        den = (p[mask] * (1 - p[mask])).sum()
        tree.tree_.value[leaf, 0, 0] = num / den if abs(den) > 1e-150 else 0.0
    return tree

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))

def gb_fit_classifier(X, y, n_estimators=100, lr=0.1, max_depth=3, track=False):
    pbar = y.mean()
    init = np.log(pbar / (1 - pbar))
    F = np.full(len(y), init); trees, losses = [], []
    for m in range(n_estimators):
        p = sigmoid(F)
        r = y - p
        tree = DecisionTreeRegressor(max_depth=max_depth, random_state=0).fit(X, r)
        newton_leaf_update(tree, X, r, p)
        F = F + lr * tree.predict(X)
        trees.append(tree)
        if track: losses.append(log_loss(y, sigmoid(F)))
    return init, trees, losses

def gb_decision_function(init, trees, lr, X):
    return init + lr * sum(t.predict(X) for t in trees)

# ✔️ Checkpoint: match sklearn's GradientBoostingClassifier (log loss)
Xm, ym = make_moons(n_samples=400, noise=0.3, random_state=0)
initc, treesc, _ = gb_fit_classifier(Xm, ym, n_estimators=50, lr=0.1, max_depth=3)
skc = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=0).fit(Xm, ym)
diff = np.abs(gb_decision_function(initc, treesc, 0.1, Xm) - skc.decision_function(Xm)).max()
print(f"max |ours - sklearn| on log-odds = {diff:.2e}")
assert diff < 1e-6
print("✔️ Your Newton-step booster matches sklearn's GradientBoostingClassifier! 🐝")

In [ ]:
initc, treesc, losses = gb_fit_classifier(Xm, ym, n_estimators=150, lr=0.2, max_depth=2, track=True)
gx, gy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 160)); G = np.c_[gx.ravel(), gy.ravel()]
fig, axes = plt.subplots(1, 5, figsize=(21, 3.8))
for ax, m in zip(axes[:4], [1, 5, 25, 150]):
    pz = sigmoid(gb_decision_function(initc, treesc[:m], 0.2, G)).reshape(gx.shape)
    ax.contourf(gx, gy, pz, levels=20, cmap="RdYlBu_r", alpha=0.7)
    ax.contour(gx, gy, pz, levels=[0.5], colors=HIVE[1], linewidths=2)
    ax.scatter(*Xm.T, c=ym, cmap="RdYlBu_r", s=10, edgecolor="k", linewidth=0.3)
    acc = ((gb_decision_function(initc, treesc[:m], 0.2, Xm) > 0) == ym).mean()
    ax.set_title(f"m={m} trees · train acc {acc:.2f}")
axes[4].plot(range(1, len(losses) + 1), losses, color=HIVE[2]); axes[4].set_xlabel("boosting round")
axes[4].set_title("training log loss"); plt.tight_layout(); plt.show()

🔍 **Notice:** the training log loss keeps going down, but by $m=150$ the boundary is getting wiggly around noisy points. That's boosting's **variance** creeping back in. Next we tame it.

> **🍯 Key takeaway:** For log loss, fit trees to $y-p$, then set each leaf to the Newton step $\sum r / \sum p(1-p)$. That gradient-over-Hessian idea is the core of XGBoost and LightGBM.

---
## Part 5 — Shrinkage, subsampling, early stopping (on real data)

Three regularizers, each with a clear story:
* **Shrinkage** ($\nu < 1$): take smaller functional-gradient steps. You need more trees, but the path is smoother and generalizes better (Friedman, 2001, found small values such as $\nu \le 0.1$ generally give the best test error).
* **Row subsampling** (*stochastic gradient boosting*, Friedman 2002): each tree sees a random fraction of rows. That decorrelates the trees, adds bagging-like variance reduction, and makes each round faster.
* **Early stopping:** watch a **validation** loss and stop when it hasn't improved for `patience` rounds. Keep the best iteration.

We'll use the **California housing** dataset (20,640 districts, predicting median house value in units of \$100k).

In [ ]:
def load_housing():
    try:
        from sklearn.datasets import fetch_california_housing
        d = fetch_california_housing(as_frame=True)
        return d.data, d.target, "California housing"
    except Exception as e:  # offline fallback so the lab never dies
        print("⚠️ Could not download California housing:", e, "-> using synthetic Friedman #1 instead")
        X, y = make_friedman1(n_samples=20640, n_features=8, noise=1.0, random_state=0)
        return pd.DataFrame(X, columns=[f"x{i}" for i in range(8)]), pd.Series(y, name="y"), "Friedman #1 (synthetic)"

X_all, y_all, DATA_NAME = load_housing()
if FAST:
    X_all, y_all = X_all.sample(4000, random_state=SEED), y_all.sample(4000, random_state=SEED)
X_tmp, X_test, y_tmp, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=SEED)
print(DATA_NAME, "| train", X_train.shape, "val", X_val.shape, "test", X_test.shape)
X_all.head()

### 🧪 Your Turn 4 — Early stopping logic
Write `early_stopping(val_losses, patience)` → `(best_iter, stop_iter)` (0-indexed). Training stops at the first iteration `i` where `i - best_iter >= patience`. If that never happens, `stop_iter = len(val_losses) - 1`.

*Hint:* a single loop that tracks the best loss seen so far and where it happened.

In [ ]:
# 🧪 Your Turn 4
def early_stopping(val_losses, patience):
    best_iter, best = 0, np.inf
    for i, loss in enumerate(val_losses):
        pass  # TODO: update best/best_iter; return (best_iter, i) once patience is exhausted
    return best_iter, len(val_losses) - 1

In [ ]:
# ✅ Solution (peek only after trying!)
def early_stopping(val_losses, patience):
    best_iter, best = 0, np.inf
    for i, loss in enumerate(val_losses):
        if loss < best:
            best, best_iter = loss, i
        elif i - best_iter >= patience:
            return best_iter, i
    return best_iter, len(val_losses) - 1

In [ ]:
# ✔️ Checkpoint
assert early_stopping([5, 4, 3, 3.5, 3.2, 3.1, 3.3, 3.4, 3.6], patience=3) == (2, 5)
assert early_stopping([5, 4, 3, 2, 1], patience=2) == (4, 4)
assert early_stopping([1, 2, 3], patience=1) == (0, 1)
print("✔️ Early stopping logic is solid.")

Now the full, upgraded booster. It uses the same loop as Part 3, plus **subsampling**, **validation tracking** and **early stopping**.

In [ ]:
class ScratchGBRegressor:
    """Squared-loss gradient boosting with shrinkage, row subsampling and early stopping."""
    def __init__(self, n_estimators=300, learning_rate=0.1, max_depth=3, subsample=1.0,
                 min_samples_leaf=1, early_stopping_rounds=None, random_state=0):
        self.n_estimators, self.learning_rate, self.max_depth = n_estimators, learning_rate, max_depth
        self.subsample, self.min_samples_leaf = subsample, min_samples_leaf
        self.early_stopping_rounds, self.random_state = early_stopping_rounds, random_state

    def fit(self, X, y, X_val=None, y_val=None):
        X, y = np.asarray(X, float), np.asarray(y, float)
        rng = np.random.default_rng(self.random_state)
        n = len(y); n_sub = max(2, int(self.subsample * n))
        self.init_ = y.mean(); F = np.full(n, self.init_)
        self.trees_, self.train_rmse_, self.val_rmse_ = [], [], []
        if X_val is not None:
            X_val, y_val = np.asarray(X_val, float), np.asarray(y_val, float)
            F_val = np.full(len(y_val), self.init_)
        for m in range(self.n_estimators):
            r = y - F
            idx = rng.choice(n, n_sub, replace=False) if self.subsample < 1 else np.arange(n)
            tree = DecisionTreeRegressor(max_depth=self.max_depth, min_samples_leaf=self.min_samples_leaf,
                                         random_state=int(rng.integers(1 << 31))).fit(X[idx], r[idx])
            F += self.learning_rate * tree.predict(X)
            self.trees_.append(tree); self.train_rmse_.append(np.sqrt(np.mean((y - F) ** 2)))
            if X_val is not None:
                F_val += self.learning_rate * tree.predict(X_val)
                self.val_rmse_.append(np.sqrt(np.mean((y_val - F_val) ** 2)))
                if self.early_stopping_rounds and m - int(np.argmin(self.val_rmse_)) >= self.early_stopping_rounds:
                    break                              # patience exhausted (same rule as early_stopping())
        if X_val is not None and self.early_stopping_rounds:
            self.best_iteration_ = int(np.argmin(self.val_rmse_))
            self.trees_ = self.trees_[: self.best_iteration_ + 1]
        return self

    def predict(self, X):
        X = np.asarray(X, float)
        return self.init_ + self.learning_rate * np.sum([t.predict(X) for t in self.trees_], axis=0)

> 🔮 **Predict-then-run:** we train with learning rates $\nu \in \{1.0, 0.3, 0.1, 0.03\}$ for the same number of trees. Which reaches the lowest validation RMSE? Which overfits first? Will $\nu = 0.03$ even finish converging?

In [ ]:
N_TREES = 60 if FAST else 400
curves, t0 = {}, time.time()
for lr in [1.0, 0.3, 0.1, 0.03]:
    m = ScratchGBRegressor(n_estimators=N_TREES, learning_rate=lr, max_depth=4).fit(X_train, y_train, X_val, y_val)
    curves[lr] = (m.train_rmse_, m.val_rmse_)
print(f"trained {4 * N_TREES} trees in {time.time() - t0:.1f}s")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for c, (lr, (tr, va)) in zip([HIVE[2], HIVE[0], HIVE[3], HIVE[4]], curves.items()):
    axes[0].plot(tr, color=c, label=f"ν={lr}"); axes[1].plot(va, color=c, label=f"ν={lr}")
    b = int(np.argmin(va)); axes[1].scatter([b], [va[b]], color=c, edgecolor="k", zorder=5)
axes[0].set_title("train RMSE"); axes[1].set_title("validation RMSE (● = best)")
for ax in axes: ax.set_xlabel("# trees"); ax.legend()
plt.tight_layout(); plt.show()
for lr, (tr, va) in curves.items():
    print(f"ν={lr:<5} best val RMSE {min(va):.4f} at {int(np.argmin(va)) + 1} trees")

In [ ]:
sub_curves = {}
for ss in [1.0, 0.5, 0.2]:
    t0 = time.time()
    m = ScratchGBRegressor(n_estimators=N_TREES, learning_rate=0.1, max_depth=4, subsample=ss, random_state=1).fit(X_train, y_train, X_val, y_val)
    sub_curves[ss] = (m.val_rmse_, time.time() - t0)
fig, ax = plt.subplots(figsize=(7, 4))
for c, (ss, (va, tt)) in zip([HIVE[1], HIVE[3], HIVE[5]], sub_curves.items()):
    ax.plot(va, color=c, label=f"subsample={ss}  ({tt:.1f}s, best {min(va):.3f})")
ax.set_xlabel("# trees"); ax.set_ylabel("validation RMSE"); ax.set_title("Stochastic gradient boosting"); ax.legend()
plt.show()

In [ ]:
# Early stopping in action
es = ScratchGBRegressor(n_estimators=5 * N_TREES, learning_rate=0.1, max_depth=4, subsample=0.8,
                        early_stopping_rounds=20 if FAST else 30, random_state=0).fit(X_train, y_train, X_val, y_val)
print(f"ran {len(es.val_rmse_)} rounds, kept best = {es.best_iteration_ + 1} trees")
print(f"test RMSE (scratch, early-stopped): {np.sqrt(mean_squared_error(y_test, es.predict(X_test))):.4f}")

> **🍯 Key takeaway:** Small learning rate + row subsampling + early stopping on a validation set is the default boosting recipe. You trade more trees for better generalization.

---
## Part 6 — Showdown: scratch vs sklearn vs HistGB vs LightGBM vs XGBoost

What do the production libraries do that we don't?

| Trick | Who | Why it helps |
|---|---|---|
| **Histogram binning** (≈255 bins/feature) | HistGB, LightGBM, XGBoost `hist` | Split finding costs $O(\text{bins})$, not $O(n \log n)$. Huge speedups. |
| **Second-order (Newton) leaves + L2** $\gamma = -G/(H+\lambda)$ | XGBoost, LightGBM, HistGB | Better steps for any twice-differentiable loss, plus built-in regularization |
| **Leaf-wise (best-first) growth** | LightGBM, HistGB (capped by `num_leaves` / `max_leaf_nodes`) | Spends splits where the loss drops the most |
| **GOSS / exclusive feature bundling** (optional) | LightGBM | Subsamples mostly the small-gradient rows; bundles mutually exclusive sparse features |
| **Native missing values & categoricals** | all three | No imputation or one-hot needed |
| **Multithreading, cache-aware layouts, GPU** | all | Engineering 🛠 |

> 🔮 **Predict-then-run:** rank the five models by test RMSE and by fit time before running.

In [ ]:
results = []
def record(name, model_fn):
    t0 = time.time(); pred = model_fn(); t = time.time() - t0
    results.append({"model": name, "test RMSE": np.sqrt(mean_squared_error(y_test, pred)), "fit+predict s": t})

MAXT = 80 if FAST else 600
record("Scratch GB (ours)", lambda: ScratchGBRegressor(n_estimators=MAXT, learning_rate=0.1, max_depth=4, subsample=0.8,
                                                       early_stopping_rounds=30, random_state=0)
                                     .fit(X_train, y_train, X_val, y_val).predict(X_test))
record("sklearn GradientBoosting", lambda: GradientBoostingRegressor(n_estimators=es.best_iteration_ + 1, learning_rate=0.1, max_depth=4,
                                                                    subsample=0.8, random_state=0).fit(X_train, y_train).predict(X_test))
record("sklearn HistGradientBoosting", lambda: HistGradientBoostingRegressor(max_iter=MAXT, learning_rate=0.1, early_stopping=True,
                                                                            random_state=0).fit(X_tmp, y_tmp).predict(X_test))
try:
    import lightgbm as lgb
    record("LightGBM", lambda: lgb.LGBMRegressor(n_estimators=4 * MAXT, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1,
                                                 colsample_bytree=0.9, verbose=-1, random_state=0)
                               .fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
                               .predict(X_test))
except Exception as e:
    print("⚠️ LightGBM unavailable:", repr(e)[:120])
try:
    import xgboost as xgb
    record("XGBoost", lambda: xgb.XGBRegressor(n_estimators=4 * MAXT, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.9,
                                               tree_method="hist", early_stopping_rounds=50, random_state=0)
                              .fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False).predict(X_test))
except Exception as e:
    print("⚠️ XGBoost unavailable:", repr(e)[:120])

res = pd.DataFrame(results).sort_values("test RMSE").reset_index(drop=True)
display(res.style.format({"test RMSE": "{:.4f}", "fit+predict s": "{:.2f}"}).background_gradient(subset=["test RMSE"], cmap="YlOrBr_r"))
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].barh(res["model"], res["test RMSE"], color=HIVE[0]); axes[0].set_title("test RMSE (lower is better)"); axes[0].invert_yaxis()
axes[1].barh(res["model"], res["fit+predict s"], color=HIVE[4]); axes[1].set_title("wall time, seconds (log)"); axes[1].set_xscale("log"); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

**Reading the table 🐝.** Your scratch booster and sklearn's classic `GradientBoostingRegressor` use the *same algorithm*, so they land close together. The histogram-based libraries are usually **both faster and a bit more accurate**. They grow bigger trees (leaf-wise, 31 leaves) on binned features, with L2-regularized Newton leaves, and they make early stopping cheap. In practice: **start with HistGradientBoosting, LightGBM, XGBoost or CatBoost**, and keep your from-scratch understanding for debugging and custom losses.

> **🍯 Key takeaway:** Same algorithm, better engineering: histogram binning, regularized second-order leaves and multithreading make production GBDTs faster *and* usually more accurate.

---
## Part 7 — Feature importance pitfalls 🕳️

Tree ensembles hand you `feature_importances_` for free. Free things deserve suspicion.

1. **Impurity-based importance (MDI)** sums the loss reduction from every split on a feature, *measured on the training data*. Features with many unique values (continuous noise, IDs) offer more candidate splits, so they pick up importance by overfitting (Strobl et al., 2007).
2. **Correlated features split the credit.** If two features carry the same signal, the model can use either one, and both look "half as important" as the signal really is.
3. **Permutation importance** = how much a held-out metric drops when you shuffle one column. It's measured on **test data**, so noise columns score ≈ 0. It still suffers from (2), and shuffling creates unrealistic rows when features are correlated.

Let's plant some traps: a pure-noise continuous column, a high-cardinality fake "ID", and a near-duplicate of the top feature.

In [ ]:
rngfi = np.random.default_rng(7)
def add_traps(X, top):
    X = X.copy()
    X["noise_cont"] = rngfi.normal(size=len(X))
    X["fake_id"] = rngfi.integers(0, 10_000, size=len(X)).astype(float)
    X[top + "_twin"] = X[top] + rngfi.normal(0, 0.05 * X[top].std(), len(X))
    return X

TOP = X_train.columns[0]          # MedInc for California housing
n_fi = 3000 if FAST else 8000
Xtr_fi, ytr_fi = add_traps(X_train.iloc[:n_fi], TOP), y_train.iloc[:n_fi]
Xte_fi = add_traps(X_test, TOP)
rf = RandomForestRegressor(n_estimators=30 if FAST else 100, min_samples_leaf=1, n_jobs=N_JOBS, random_state=0).fit(Xtr_fi, ytr_fi)
print(f"RF test R² = {r2_score(y_test, rf.predict(Xte_fi)):.3f}")

### 🧪 Your Turn 5 — Permutation importance from scratch
For each column: shuffle it (only that column) in a copy of `X`, recompute R², and record the drop from the baseline. Average over `n_repeats` shuffles.

*Hints:* `Xp = X.copy(); Xp[col] = rng.permutation(Xp[col].values)`; `r2_score(y, model.predict(Xp))`.

In [ ]:
# 🧪 Your Turn 5
def permutation_importance_scratch(model, X, y, n_repeats=5, seed=0):
    rng = np.random.default_rng(seed)
    base = r2_score(y, model.predict(X))
    imps = np.zeros((X.shape[1], n_repeats))
    for j, col in enumerate(X.columns):
        for k in range(n_repeats):
            pass  # TODO: shuffle column `col`, score, store base - score in imps[j, k]
    return imps.mean(1), imps.std(1)

In [ ]:
# ✅ Solution (peek only after trying!)
def permutation_importance_scratch(model, X, y, n_repeats=5, seed=0):
    rng = np.random.default_rng(seed)
    base = r2_score(y, model.predict(X))
    imps = np.zeros((X.shape[1], n_repeats))
    for j, col in enumerate(X.columns):
        for k in range(n_repeats):
            Xp = X.copy()
            Xp[col] = rng.permutation(Xp[col].values)
            imps[j, k] = base - r2_score(y, model.predict(Xp))
    return imps.mean(1), imps.std(1)

In [ ]:
from sklearn.inspection import permutation_importance
Xte_small, yte_small = Xte_fi.iloc[:2000], y_test.iloc[:2000]
pi_mean, pi_std = permutation_importance_scratch(rf, Xte_small, yte_small, n_repeats=3 if FAST else 5)
sk_pi = permutation_importance(rf, Xte_small, yte_small, n_repeats=3 if FAST else 5, random_state=0, n_jobs=N_JOBS)
corr = np.corrcoef(pi_mean, sk_pi.importances_mean)[0, 1]
print(f"correlation(ours, sklearn) = {corr:.4f}")
# ✔️ Checkpoint
assert corr > 0.95 and np.argmax(pi_mean) == np.argmax(sk_pi.importances_mean)
print("✔️ Your permutation importance agrees with sklearn.inspection.permutation_importance")

imp = pd.DataFrame({"MDI (train, impurity)": rf.feature_importances_, "permutation (test)": pi_mean}, index=Xtr_fi.columns)
imp = imp / imp.abs().sum()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
order = imp["permutation (test)"].sort_values().index
traps = {"noise_cont", "fake_id", TOP + "_twin"}
for ax, col, c in zip(axes, imp.columns, [HIVE[2], HIVE[3]]):
    colors = [HIVE[1] if i in traps else c for i in order]
    ax.barh(order, imp.loc[order, col], color=colors); ax.set_title(col); ax.axvline(0, color="gray", lw=0.5)
plt.suptitle("Trap features in black: MDI credits pure noise; the twin splits credit with " + TOP); plt.tight_layout(); plt.show()
display(imp.loc[["noise_cont", "fake_id", TOP, TOP + "_twin"]].round(3))

**What to notice**
* MDI gives `noise_cont` and `fake_id` a clearly **nonzero** share, often comparable to genuine but weak features like `Population`. Fully grown trees split on them to memorize training noise. Permutation importance on held-out data puts them at ≈ 0 (sometimes slightly negative). ✅ The bias gets worse with deeper trees, fewer samples and more unique values.
* `TOP` and its twin **share** the credit under both methods. Permutation importance can even make *both* look unimportant, because shuffling one leaves the other to carry the signal. The fix: permute **groups** of correlated features together, or cluster features first. sklearn's docs show how with hierarchical clustering on Spearman correlations.
* None of these are **causal** statements. They describe the *model*, not the world. We'll go deeper with SHAP in Lab 1.3.

### 🧪 Your Turn 6 — Grouped permutation importance
Shuffle `TOP` and `TOP_twin` **together** (the same row permutation for both columns) and compare the R² drop with shuffling either one alone.

In [ ]:
# 🧪 Your Turn 6
def grouped_perm_drop(model, X, y, cols, n_repeats=5, seed=0):
    rng = np.random.default_rng(seed)
    base = r2_score(y, model.predict(X))
    drops = []
    for _ in range(n_repeats):
        Xp = X.copy()
        perm = None  # TODO: one permutation of row indices, applied to every column in `cols`
        # TODO: for c in cols: Xp[c] = X[c].values[perm]
        drops.append(0.0)  # TODO: base - r2 on Xp
    return float(np.mean(drops))

In [ ]:
# ✅ Solution (peek only after trying!)
def grouped_perm_drop(model, X, y, cols, n_repeats=5, seed=0):
    rng = np.random.default_rng(seed)
    base = r2_score(y, model.predict(X))
    drops = []
    for _ in range(n_repeats):
        Xp = X.copy()
        perm = rng.permutation(len(X))
        for c in cols:
            Xp[c] = X[c].values[perm]
        drops.append(base - r2_score(y, model.predict(Xp)))
    return float(np.mean(drops))

In [ ]:
# ✔️ Checkpoint: grouped vs single-column permutation
solo_a = grouped_perm_drop(rf, Xte_small, yte_small, [TOP])
solo_b = grouped_perm_drop(rf, Xte_small, yte_small, [TOP + "_twin"])
both = grouped_perm_drop(rf, Xte_small, yte_small, [TOP, TOP + "_twin"])
print(f"R² drop: {TOP} alone {solo_a:.3f} | twin alone {solo_b:.3f} | both together {both:.3f}")
assert both > max(solo_a, solo_b), "Shuffling the group should hurt more than either alone"
print("✔️ The group's joint importance is much larger than either member's: correlated credit, recovered.")

> **🍯 Key takeaway:** Impurity (MDI) importances are computed on training data and reward high-cardinality noise. Use held-out permutation importance, and permute correlated features as a group.

---
## 🏆 Boss Level

1. **XGBoost from scratch (lite).** Replace `DecisionTreeRegressor` with your own depth-limited tree. It should split on **histogram bins** (e.g., 64 quantile bins/feature) using the second-order gain
   $\text{gain} = \tfrac12\Big[\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\Big] - \gamma$, with leaves $w = -G/(H+\lambda)$. Plug in log loss and compare with `xgboost` on the moons data.
2. **Any loss you like.** Generalize `ScratchGBRegressor` to take a `loss` object with `gradient` / `hessian` / `init` methods. Implement **Huber** and **quantile (pinball)** losses, then check robustness by planting 2% huge outliers in `y_train`.
3. **Custom objective in production.** Write a **focal loss** (Lin et al., 2017) gradient and Hessian for LightGBM's or XGBoost's custom-objective API. Train on an imbalanced `make_classification(weights=[0.97])` problem and compare PR-AUC with plain log loss.

## 🍯 Key Takeaways
* **Error = bias² + variance + noise.** Bagging fights variance; boosting fights bias.
* **Double descent:** the $p \approx n$ peak comes from unregularized interpolation; regularization flattens it.
* **Boosting = functional gradient descent.** Fit trees to negative gradients, add them with shrinkage $\nu$.
* **Log loss → Newton leaves** $\sum r/\sum p(1-p)$; XGBoost adds L2: $-G/(H+\lambda)$.
* **Regularize with the trio:** small $\nu$, subsampling, early stopping on held-out data.
* **Use the production libraries** (HistGB, LightGBM, XGBoost, CatBoost) and keep your from-scratch intuition for debugging and custom losses.
* **Never trust MDI importances blindly.** Permutation importance on held-out data, grouped for correlated features.

## 🎤 Explain it to a friend
Without any jargon, explain to a friend why a *team* of tiny, dumb decision trees, each one fixing the mistakes the team still makes, can beat one big clever tree. Try a golf analogy: each putt corrects the miss of the previous one. Then explain why the team should take *small* putts (learning rate) and stop before it starts "correcting" random bumps in the green (early stopping).

## 💼 Interview corner
**Q1. Why does gradient boosting usually use shallow trees, while random forests use deep ones?**
<details><summary>Answer</summary>

Bias–variance. A random forest *averages* independent-ish trees, which reduces variance but not bias, so each tree must be low-bias (deep). Boosting reduces bias *sequentially*, so each learner can be high-bias/low-variance (shallow). Deep trees would overfit within a few rounds. Depth also caps the interaction order: a depth-$d$ tree can model at most $d$-way interactions.
</details>

**Q2. What are the pseudo-residuals for log loss, and why do XGBoost and LightGBM also need the Hessian?**
<details><summary>Answer</summary>

With $p=\sigma(F)$, the negative gradient is $y - p$. The Hessian $p(1-p)$ gives second-order (Newton) leaf values $w = -G/(H+\lambda)$ and the split-gain formula $\tfrac12[G_L^2/(H_L+\lambda) + G_R^2/(H_R+\lambda) - (G_L+G_R)^2/(H_L+H_R+\lambda)] - \gamma$. Because of this, any twice-differentiable loss can be plugged in through just its gradient and Hessian.
</details>

**Q3. Your model's top `feature_importances_` entry is `customer_id`. What's going on, and how do you check?**
<details><summary>Answer</summary>

Impurity-based importance is measured on training data and favours high-cardinality features, which trees can use to memorize. Check permutation (or drop-column) importance on a held-out set. If the ID still matters, suspect **leakage**: IDs often encode time or a data-collection process correlated with the label.
</details>

## 🐝 Fun fact
Boosting began as a theory question. Around 1988–89, Michael Kearns and Leslie Valiant asked whether "weak" learners, only slightly better than random guessing, could be combined into an arbitrarily accurate "strong" learner. Robert Schapire proved the answer is yes (*The Strength of Weak Learnability*, 1990), and Yoav Freund and Schapire's AdaBoost later earned them the 2003 Gödel Prize.

## 📚 Further reading
* J. H. Friedman (2001). *Greedy Function Approximation: A Gradient Boosting Machine.* Annals of Statistics 29(5).
* J. H. Friedman (2002). *Stochastic Gradient Boosting.* Computational Statistics & Data Analysis 38(4).
* L. Breiman (1996). *Bagging Predictors.* Machine Learning 24.
* T. Chen & C. Guestrin (2016). *XGBoost: A Scalable Tree Boosting System.* KDD.
* G. Ke et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision Tree.* NeurIPS.
* L. Prokhorenkova et al. (2018). *CatBoost: unbiased boosting with categorical features.* NeurIPS.
* M. Belkin, D. Hsu, S. Ma, S. Mandal (2019). *Reconciling modern machine-learning practice and the classical bias–variance trade-off.* PNAS 116(32).
* P. Nakkiran et al. (2020). *Deep Double Descent: Where Bigger Models and More Data Hurt.* ICLR.
* C. Strobl, A.-L. Boulesteix, A. Zeileis, T. Hothorn (2007). *Bias in random forest variable importance measures: Illustrations, sources and a solution.* BMC Bioinformatics 8.
* L. Grinsztajn, E. Oyallon, G. Varoquaux (2022). *Why do tree-based models still outperform deep learning on typical tabular data?* NeurIPS Datasets & Benchmarks.
* R. E. Schapire (1990). *The Strength of Weak Learnability.* Machine Learning 5.
* Y. Freund & R. E. Schapire (1997). *A Decision-Theoretic Generalization of On-Line Learning and an Application to Boosting.* JCSS 55.
* T. Hastie, R. Tibshirani, J. Friedman (2009). *The Elements of Statistical Learning*, 2nd ed., Ch. 7 (model assessment) & Ch. 10 (boosting).

🐝 *Buzz says: nice work. You've built the workhorse of tabular ML from scratch. Next up in Lab 1.2: can we **trust** its probabilities?*